In [1]:
from lxml import etree
from pathlib import Path
import random
import pandas as pd

from medical_rag.text_chunker import PubMedFullTextChunker
from medical_rag.config import parse_pmc

e:\anaconda3\envs\medrag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 创建Metadata dataframe

In [2]:
xml_files = list(
    Path(
        r"E:\字节跳动算法实习\data\PMC\extracted"
    ).rglob("*.xml")
)

print(len(xml_files))

sample_files = random.sample(
    xml_files,
    30000
)

3242816


In [ ]:
# 创建dataframe来记录文献metadata

# 每批处理10000篇文献
batch_size = 10000

# 保存目录
output_dir = Path("F:/RAG/data/parsed_batches")
output_dir.mkdir(exist_ok=True)

records = []
batch_id = 0

for i, file in enumerate(sample_files):

    try:
        records.append(parse_pmc(file))

    except Exception as e:
        print(f"Skip {file}: {e}")

    # 满一个batch或者处理到最后一个文件
    if len(records) == batch_size or i == len(sample_files) - 1:

        # 创建DataFrame
        df_batch = pd.DataFrame(records)

        # 保存
        output_file = output_dir / f"batch_{batch_id:04d}.parquet"
        df_batch.to_parquet(output_file, index=False)

        print(f"Batch {batch_id} saved: {len(df_batch)} documents")

        # 清空内存
        records.clear()

        batch_id += 1

<>:7: SyntaxWarning: invalid escape sequence '\d'
<>:7: SyntaxWarning: invalid escape sequence '\d'
C:\Users\xukeh\AppData\Local\Temp\ipykernel_32116\3551698894.py:7: SyntaxWarning: invalid escape sequence '\d'
  output_dir = Path("E:\字节跳动算法实习\data\parsed_batches")


Batch 0 saved: 10000 documents
Batch 1 saved: 10000 documents
Batch 2 saved: 10000 documents


In [3]:
# Parquet目录
parquet_dir = Path("F:/RAG/data/parsed_batches")

# 获取所有Parquet文件
parquet_files = sorted(parquet_dir.glob("*.parquet"))

print(f"Found {len(parquet_files)} parquet files.")

# 读取并合并
df = pd.concat(
    [pd.read_parquet(f) for f in parquet_files],
    ignore_index=True
)

print(df.shape)
df.head()

Found 3 parquet files.
(30000, 6)


,title,abstract,journal,pmid,pub_date,full_text
0,Common mental disorders and subjective well-be...,IntroductionThe prevalence of common mental di...,PLoS ONE,30731006,2019,"In the field of positive psychology, well-bein..."
1,Magnetization relaxation in the single-ion mag...,\nQuantum tunneling and relaxation of magnetiz...,Physical Chemistry Chemical Physics,29671443,2018,Hiding atomic clusters behind the walls of pro...
2,A New Method of Treating the Fistula Lachrymalis,,The London Medical Journal,29139655,1781,
3,Persistence of Transmitted HIV-1 Drug Resistan...,Transmission of drug-resistant pathogens prese...,PLoS Pathogens,25798934,2015,Drug-resistant pathogens represent one of the ...
4,Prenatal Exposure to Lipopolysaccharide Combin...,BackgroundAdult metabolic syndrome may in part...,PLoS ONE,24498431,2014,The pathophysiology of Type 2 diabetes (T2DM) ...


## 初始化文本分割器

In [4]:
#初始化文本分割器

chunker = PubMedFullTextChunker(
    tokenizer_name="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    chunk_size=500,
    chunk_overlap=80,
    min_text_tokens=30,
    max_token_limit=512,
    add_title_to_each_chunk=True,
)

chunks_df, stats = chunker.split_dataframe(
    df_raw=df,
    full_text_col="full_text",
    output_path="E:\字节跳动算法实习\data\pubmed_fulltext_chunks.parquet",
    stats_path="E:\字节跳动算法实习\data\pubmed_fulltext_chunk_stats.json",
    data_split="pmc_full_text"
)

print(stats)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (21839 > 16384). Running this sequence through the model will result in indexing errors
Splitting documents:   2%|▏         | 429/28246 [01:11<1:17:34,  5.98it/s]


KeyboardInterrupt: 

## 预览分割结果

In [5]:
chunks_df = pd.read_parquet(
    "F:\RAG\data\pubmed_fulltext_chunks_2.parquet"
)

print(chunks_df.shape)
chunks_df.head()

(668461, 11)


,chunk_id,text,doc_id,chunk_index,total_chunks,source_title,journal,pub_date,pmid,token_count,split_strategy
0,PMID_30731006_chunk_0000,Title: Common mental disorders and subjective ...,PMID_30731006,0,17,Common mental disorders and subjective well-be...,PLoS ONE,2019,30731006,500,sliding_window
1,PMID_30731006_chunk_0001,Title: Common mental disorders and subjective ...,PMID_30731006,1,17,Common mental disorders and subjective well-be...,PLoS ONE,2019,30731006,487,sliding_window
2,PMID_30731006_chunk_0002,Title: Common mental disorders and subjective ...,PMID_30731006,2,17,Common mental disorders and subjective well-be...,PLoS ONE,2019,30731006,503,sliding_window
3,PMID_30731006_chunk_0003,Title: Common mental disorders and subjective ...,PMID_30731006,3,17,Common mental disorders and subjective well-be...,PLoS ONE,2019,30731006,491,sliding_window
4,PMID_30731006_chunk_0004,Title: Common mental disorders and subjective ...,PMID_30731006,4,17,Common mental disorders and subjective well-be...,PLoS ONE,2019,30731006,491,sliding_window


In [6]:
chunks_df.columns

Index(['chunk_id', 'text', 'doc_id', 'chunk_index', 'total_chunks',
       'source_title', 'journal', 'pub_date', 'pmid', 'token_count',
       'split_strategy'],
      dtype='object')

In [7]:
#chunk数量统计
print("Total chunks:", len(chunks_df))
print(
    "Total documents:",
    chunks_df["doc_id"].nunique()
)
print(
    "Average chunks/document:",
    len(chunks_df)
    /
    chunks_df["doc_id"].nunique()
)

Total chunks: 668461
Total documents: 28246
Average chunks/document: 23.665687176945408


In [8]:
#Token长度检查
print(
    "Maximum:",
    chunks_df["token_count"].max()
)

print(
    "95 percentile:",
    chunks_df["token_count"].quantile(0.95)
)
print(
    "Chunks with more than 512 tokens:",
    (chunks_df["token_count"] > 512).sum()
)

Maximum: 513
95 percentile: 512.0
Chunks with more than 512 tokens: 3


In [9]:
#split_strategy统计
chunks_df["split_strategy"].value_counts()

split_strategy
sliding_window    667879
no_split             582
Name: count, dtype: int64

## 多块文档检查

In [10]:
# 找出被分割成多个 chunk 的文献
multi_doc_ids = (
    chunks_df[chunks_df["total_chunks"] > 1]["doc_id"]
    .drop_duplicates()
)

print("Number of multi-chunk documents:", len(multi_doc_ids))
print("Rate:", len(multi_doc_ids) / chunks_df["doc_id"].nunique() * 100)

Number of multi-chunk documents: 27664
Rate: 97.93953126106352


In [11]:
#检查 chunk_index 是否连续
def check_chunk_index_continuity(group):
    indices = sorted(group["chunk_index"].tolist())
    expected = list(range(len(indices)))
    return indices == expected

continuity_report = (
    chunks_df.groupby("doc_id")
    .apply(check_chunk_index_continuity)
)

print("Documents with continuous chunk_index:", continuity_report.sum())
print("Documents with broken chunk_index:", (~continuity_report).sum())

Documents with continuous chunk_index: 28246
Documents with broken chunk_index: 0


C:\Users\xukeh\AppData\Local\Temp\ipykernel_27620\4017873602.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(check_chunk_index_continuity)


In [12]:
sample_multi_docs = multi_doc_ids.sample(
    min(5, len(multi_doc_ids)),
    random_state=42
)

for doc_id in sample_multi_docs:
    doc_chunks = (
        chunks_df[chunks_df["doc_id"] == doc_id]
        .sort_values("chunk_index")
    )

    print("=" * 120)
    print("doc_id:", doc_id)
    print("source_title:", doc_chunks.iloc[0]["source_title"])
    print("total_chunks:", doc_chunks.iloc[0]["total_chunks"])

    for _, row in doc_chunks.head(3).iterrows():
        print("-" * 80)
        print("chunk_index:", row["chunk_index"])
        print("chunk_id:", row["chunk_id"])
        print("token_count:", row["token_count"])
        print("text start:")
        print(row["text"][:300])
        print("text end:")
        print(row["text"][-300:])

doc_id: DOC_d4ddb8a8944b
source_title: Anti-interleukin 1 treatment in secondary renal amyloidosis associated with autoinflammatory diseases
total_chunks: 2
--------------------------------------------------------------------------------
chunk_index: 0
chunk_id: DOC_d4ddb8a8944b_chunk_0000
token_count: 512
text start:
Title: Anti-interleukin 1 treatment in secondary renal amyloidosis associated with autoinflammatory diseases

Amyloidosis represents a heterogeneous group of disorders characterized by extracellular deposition of autologous fibrillary proteins which impair normal organ function. Reactive AA type amy
text end:
ment of amyloidosis, it is crucial to control inflammation effectively and prevent further amyloid accumulation in patients with anti-inflammatory treatments such as anti-IL1 drugs. However, new treatment strategies are needed to target the amyloid deposits for patients with severe organ involvement
--------------------------------------------------------------------

In [13]:
#检查 total_chunks 是否与实际 chunk 数一致
def check_total_chunks_consistency(group):
    actual = len(group)
    recorded = group["total_chunks"].iloc[0]
    return actual == recorded

total_chunks_report = (
    chunks_df.groupby("doc_id")
    .apply(check_total_chunks_consistency)
)

print("Documents with correct total_chunks:", total_chunks_report.sum())
print("Documents with inconsistent total_chunks:", (~total_chunks_report).sum())

Documents with correct total_chunks: 28246
Documents with inconsistent total_chunks: 0


C:\Users\xukeh\AppData\Local\Temp\ipykernel_27620\766120934.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(check_total_chunks_consistency)


In [14]:
#检查标题是否保留
title_samples = chunks_df.sample(10, random_state=42)

for _, row in title_samples.iterrows():
    print("=" * 100)
    print("chunk_id:", row["chunk_id"])
    print("doc_id:", row["doc_id"])
    print("source_title:", row["source_title"])
    print("text preview:")
    print(row["text"][:500])

chunk_id: PMID_35514340_chunk_0003
doc_id: PMID_35514340
source_title: Surfactant Treatment Shows Higher Correlation Between Ventilator and EIT Tidal Volumes in an RDS Animal Model
text preview:
Title: Surfactant Treatment Shows Higher Correlation Between Ventilator and EIT Tidal Volumes in an RDS Animal Model

. Mann-Whitney U test was used in between-group comparisons. Statistical significance was set at p < 0.05 and SAS version 9.4 (SAS Institute Inc., Cary, NC, United States) as well as the R4.0.4 program was used for analyses. Twenty three newborn pups were harvested from 6 mother rabbits. Five pups were excluded as a result of early death (n = 2) or measurement failure due to pneu
chunk_id: PMID_34947312_chunk_0014
doc_id: PMID_34947312
source_title: Evaluation of Functional Features of Lignocellulosic Particle Composites Containing Biopolymer Binders
text preview:
Title: Evaluation of Functional Features of Lignocellulosic Particle Composites Containing Biopolymer Binders

. The

## 检查重叠窗口

In [15]:
# 找出多块文档
multi_doc_ids = (
    chunks_df[chunks_df["total_chunks"] > 1]["doc_id"]
    .drop_duplicates()
    .tolist()
)

# 随机抽取 5 篇多块文档
sample_doc_ids = random.sample(
    multi_doc_ids,
    k=min(5, len(multi_doc_ids))
)

for doc_id in sample_doc_ids:
    doc_chunks = (
        chunks_df[chunks_df["doc_id"] == doc_id]
        .sort_values("chunk_index")
        .reset_index(drop=True)
    )

    print("=" * 120)
    print("doc_id:", doc_id)
    print("source_title:", doc_chunks.loc[0, "source_title"])
    print("total_chunks:", doc_chunks.loc[0, "total_chunks"])

    # 每篇文档最多检查前 3 对相邻 chunk
    max_pairs = min(3, len(doc_chunks) - 1)

    for i in range(max_pairs):
        chunk_a = doc_chunks.loc[i, "text"]
        chunk_b = doc_chunks.loc[i + 1, "text"]

        print("-" * 100)
        print(f"Chunk {i} END:")
        print(chunk_a[-800:])

        print("\n")
        print(f"Chunk {i + 1} START:")
        print(chunk_b[:800])

doc_id: PMID_25003337
source_title: Exposure to Silica Nanoparticles Causes Reversible Damage of the Spermatogenic Process in Mice
total_chunks: 31
----------------------------------------------------------------------------------------------------
Chunk 0 END:
oth the testis and prostate in male offsprings and disrupted the endocrine activity of the male reproductive system by decreasing the levels of testosterone, progesterone and corticosterone in the serum [6]. However, Ramdhan et al [7] found that the exposure to a low concentration of diesel exhaust rich in nanoparticles increased the levels of plasma testosterone, which contradict to Li et al.'s report [6]. The human male reproductive system is known to be vulnerable to many exogenous materials [8]. Additionally, oxidative stress is known to be one of the main mechanisms of this deterioration [9]. Others and we have attributed silica nanoparticle-induced cytotoxicity to oxidative stress [10]–[12]. However, the effects of silica 

In [16]:
#检查空文本
print("Chunks with empty text:", (
    chunks_df["text"]
    .astype(str)
    .str.strip()
    .eq("")
).sum())

Chunks with empty text: 0


In [17]:
#检查token长度小于40的chunk
chunks_df[
    chunks_df["token_count"]<40
]

short = chunks_df[
    chunks_df["token_count"]<40
]

print(len(short))
short.sample(5)

25


,chunk_id,text,doc_id,chunk_index,total_chunks,source_title,journal,pub_date,pmid,token_count,split_strategy
229528,PMID_30800380_chunk_0018,Title: Orientation-invariance of individual di...,PMID_30800380,18,36,Orientation-invariance of individual differenc...,Royal Society Open Science,2019,30800380,35,sliding_window
213170,PMID_30430418_chunk_0029,Title: The Combined Effects of Cr(III) Supplem...,PMID_30430418,29,50,The Combined Effects of Cr(III) Supplementatio...,Biological Trace Element Research,2018,30430418,36,sliding_window
159161,PMID_30911046_chunk_0040,Title: On the stress-strain alignment in premi...,PMID_30911046,40,59,On the stress-strain alignment in premixed tur...,Scientific Reports,2019,30911046,37,sliding_window
640132,DOC_3dc7c7b36355_chunk_0015,Title: Research on fault diagnosis method of p...,DOC_3dc7c7b36355,15,33,Research on fault diagnosis method of planetar...,Scientific Reports,2022,,38,sliding_window
268199,PMID_36508026_chunk_0029,Title: Decentralized collaborative multi-insti...,PMID_36508026,29,54,Decentralized collaborative multi-institutiona...,European Journal of Nuclear Medicine and Molec...,2022,36508026,39,sliding_window


In [18]:
#检查是否截断严重
sample = chunks_df.sample(
    10,
    random_state=42
)

for _, row in sample.iterrows():

    print("="*80)

    print(
        row["chunk_id"]
    )

    print("\nBEGIN:")

    print(
        row["text"][:250]
    )

    print("\nEND:")

    print(
        row["text"][-250:]
    )

PMID_35514340_chunk_0003

BEGIN:
Title: Surfactant Treatment Shows Higher Correlation Between Ventilator and EIT Tidal Volumes in an RDS Animal Model

. Mann-Whitney U test was used in between-group comparisons. Statistical significance was set at p < 0.05 and SAS version 9.4 (SAS I

END:
es are shown in Table 1. * means statistical significance between the untreated preterm group and surfactant treated preterm group (p = 0.018, 0.026, Mann-Whitney U test). PIP, peak inspiratory pressure; Inf, inflation; Def, deflation. An increase in
PMID_34947312_chunk_0014

BEGIN:
Title: Evaluation of Functional Features of Lignocellulosic Particle Composites Containing Biopolymer Binders

. The density of surface layers became higher in the core when the RC of the particleboard increased. The modulus of rupture (MOR) and modu

END:
 sample does not experience shear failure during the static bending test [47]. According to statistical analysis, there are no significant differences between the averag

In [19]:
#检查overlap是否正常
doc = chunks_df[
    chunks_df["doc_id"]==doc_id
].sort_values("chunk_index")

for i in range(len(doc)-1):

    print("="*80)

    print("Chunk",i)

    print(doc.iloc[i]["text"][-200:])

    print()

    print("Chunk",i+1)

    print(doc.iloc[i+1]["text"][:200])

Chunk 0
ve ammonia injection15. More, these factors can all lead to an increase in ammonia escape. The escaped ammonia will promote the production of more ABS, and ABS is an important factor in catalyst block

Chunk 1
same time, to ensure a higher denitrification efficiency, the amount of ammonia injection is often increased, or the ammonia injection is not adjusted in time when the load fluctuates, which may easil
Chunk 1
re, these factors can all lead to an increase in ammonia escape. The escaped ammonia will promote the production of more ABS, and ABS is an important factor in catalyst blockage and decreased activity

Chunk 2
Title: Status and development for detection and control of ammonium bisulfate as a by-product of SCR denitrification

. More, these factors can all lead to an increase in ammonia escape. The escaped a
Chunk 2
y measured the ammonium concentration by spectrophotometry18. However, due to the complexity of the composition of the ammonium sulfate salt, the results 

In [20]:
#最终质量评估报告
quality_report = {
    "Total Documents": chunks_df["doc_id"].nunique(),
    "Total Chunks": len(chunks_df),
    "Average Chunks per Document": len(chunks_df) / chunks_df["doc_id"].nunique(),
    "Maximum Token Count": chunks_df["token_count"].max(),
    "Chunks >512 Tokens": (chunks_df["token_count"] > 512).sum(),
    "Empty Chunks": (
        chunks_df["text"]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    ),
    "Documents Split": (
        chunks_df.groupby("doc_id")
        .first()["total_chunks"]
        .gt(1)
        .sum()
    )
}

pd.DataFrame(
    quality_report.items(),
    columns=["Metric", "Value"]
)

,Metric,Value
0,Total Documents,28246.000000
1,Total Chunks,668461.000000
2,Average Chunks per Document,23.665687
3,Maximum Token Count,513.000000
4,Chunks >512 Tokens,3.000000
5,Empty Chunks,0.000000
6,Documents Split,27664.000000
